In [ ]:
!pip install -q chromadb sentence-transformers pandas

import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))

movies = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Total movies:", len(movies))
print("Columns:", movies.columns.tolist())


model = SentenceTransformer("all-MiniLM-L6-v2")
descriptions = movies["Description"].fillna("").tolist()

embeddings = model.encode(descriptions)

print("Embedding shape:", embeddings.shape)

# Create ChromaDB collection
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="movies",
    metadata={"hnsw:space": "cosine"}
)

# Insert movie details into ChromaDB
for index, movie in movies.iterrows():

    collection.upsert(
        ids=[str(index)],
        documents=[movie["Description"]],
        embeddings=[embeddings[index].tolist()],
        metadatas=[{
            "title": str(movie["Title"]),
            "genre": str(movie["Genre"]),
            "director": str(movie["Director"])
        }]
    )

print("Movies inserted successfully!")
print("Total records:", collection.count())


#Create metadata filter
def create_filter(query):

    conditions = []

    for genre in movies["Genre"].dropna().unique():
        if str(genre).lower() in query.lower():
            conditions.append({"genre": str(genre)})

    for director in movies["Director"].dropna().unique():
        if str(director).lower() in query.lower():
            conditions.append({"director": str(director)})

    if len(conditions) == 1:
        return conditions[0]

    if len(conditions) > 1:
        return {"$and": conditions}

    return None



user_query = input("Enter your movie preference: ")
query_embedding = model.encode([user_query])[0].tolist()

where_filter = create_filter(user_query)
search_options = {
    "query_embeddings": [query_embedding],
    "n_results": 5,
    "include": ["documents", "metadatas", "distances"]
}

if where_filter is not None:
    search_options["where"] = where_filter


results = collection.query(**search_options)

print("\n========== MOVIE RECOMMENDATIONS ==========\n")

if results["metadatas"][0]:

    for index, movie in enumerate(results["metadatas"][0], start=1):

        print(f"Recommendation {index}")
        print("Title:", movie["title"])
        print("Genre:", movie["genre"])
        print("Director:", movie["director"])
        print("------------------------------")

else:
    print("No matching movies found.")

print("\nSearch completed!")

print("Distance:", results["distances"][0][index - 1])


Saving Movies Dataset - Sheet1.csv to Movies Dataset - Sheet1 (2).csv
Dataset loaded successfully!
Total movies: 32
Columns: ['Title', 'Description', 'Genre', 'Director']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (32, 384)
Movies inserted successfully!
Total records: 32
Enter your movie preference: comedy

========== MOVIE RECOMMENDATIONS ==========

Recommendation 1
Title: The Big Lebowski
Genre: Comedy
Director: Coen Brothers
------------------------------

Search completed!
Distance: 0.8193860054016113


In [ ]:
print("Total records:", collection.count())

test_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

print("Number of results:", len(test_results["ids"][0]))

for movie in test_results["metadatas"][0]:
    print(movie["title"], "-", movie["genre"])

Total records: 32
Number of results: 5
The Sixth Sense - Supernatural Thriller
The Silence of the Lambs - Horror
Inception - Science Fiction
Oldboy - Neo-Noir
The Lord of the Rings: The Fellowship of the Ring - Fantasy
